# 4.1 FER2013 Split Creation

Purpose:
- Load FER2013 manifest
- Preserve official test split
- Create stratified train/validation split from original train data
- Save processed split manifest

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    FER2013_INTERIM_MANIFEST_PATH,
    FER2013_SPLIT_MANIFEST_PATH,
)

In [2]:
fer_df = pd.read_csv(FER2013_INTERIM_MANIFEST_PATH)

fer_df.head()

,path,file_name,class_label,label_encoded,split,extension,readable,width,height
0,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,PrivateTest_10131363.jpg,angry,0,test,.jpg,True,48,48
1,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,PrivateTest_10304478.jpg,angry,0,test,.jpg,True,48,48
2,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,PrivateTest_1054527.jpg,angry,0,test,.jpg,True,48,48
3,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,PrivateTest_10590091.jpg,angry,0,test,.jpg,True,48,48
4,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,PrivateTest_1109992.jpg,angry,0,test,.jpg,True,48,48


In [3]:
original_train_df = fer_df[
    (fer_df["split"] == "train")
    & (fer_df["readable"] == True)
    & (fer_df["class_label"] != "unknown")
].copy()

test_df = fer_df[
    (fer_df["split"] == "test")
    & (fer_df["readable"] == True)
    & (fer_df["class_label"] != "unknown")
].copy()

len(original_train_df), len(test_df)

(28709, 7178)

In [4]:
train_df, val_df = train_test_split(
    original_train_df,
    test_size=0.15,
    stratify=original_train_df["class_label"],
    random_state=42,
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

fer_split_df = pd.concat(
    [train_df, val_df, test_df],
    axis=0,
).reset_index(drop=True)

fer_split_df.groupby(["split", "class_label"]).size()

split  class_label
test   angry           958
       disgust         111
       fear           1024
       happy          1774
       neutral        1233
       sad            1247
       surprise        831
train  angry          3396
       disgust         371
       fear           3482
       happy          6133
       neutral        4220
       sad            4105
       surprise       2695
val    angry           599
       disgust          65
       fear            615
       happy          1082
       neutral         745
       sad             725
       surprise        476
dtype: int64

In [5]:
FER2013_SPLIT_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

fer_split_df.to_csv(FER2013_SPLIT_MANIFEST_PATH, index=False)

FER2013_SPLIT_MANIFEST_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/data/processed/fer2013_split_manifest.csv')

## Split Decision

The original FER2013 test set is preserved. A validation split is created from the original training set using stratified sampling so that class proportions remain stable across train and validation.